In [1]:
!pip install pandas numpy scikit-learn imbalanced-learn xgboost joblib

In [3]:
#!/bin/bash
!kaggle datasets download primus11/nsl-kdd-dataset-filtered-version-of-kdd

Dataset URL: https://www.kaggle.com/datasets/primus11/nsl-kdd-dataset-filtered-version-of-kdd
License(s): MIT
100% 6.93M/6.93M [00:00<00:00, 34.3MB/s]



In [4]:
#unzip the file
!unzip -q nsl-kdd-dataset-filtered-version-of-kdd.zip

In [5]:
!ls -lah

total 61M
drwxr-xr-x 1 root root 4.0K May 23 15:41 .
drwxr-xr-x 1 root root 4.0K May 23 14:55 ..
drwxr-xr-x 4 root root 4.0K May 21 13:26 .config
-rw-r--r-- 1 root root  33K Aug 13  2025 index.html
-rw-r--r-- 1 root root 8.5K Aug 13  2025 KDDTest1.jpg
-rw-r--r-- 1 root root 1.7M Aug 13  2025 KDDTest-21.arff
-rw-r--r-- 1 root root 1.8M Aug 13  2025 KDDTest-21.txt
-rw-r--r-- 1 root root 3.3M Aug 13  2025 KDDTest.arff
-rw-r--r-- 1 root root 3.3M Aug 13  2025 KDDTest.txt
-rw-r--r-- 1 root root 8.4K Aug 13  2025 KDDTrain1.jpg
-rw-r--r-- 1 root root 3.6M Aug 13  2025 KDDTrain_20Percent.arff
-rw-r--r-- 1 root root 3.7M Aug 13  2025 KDDTrain_20Percent.txt
-rw-r--r-- 1 root root  18M Aug 13  2025 KDDTrain.arff
-rw-r--r-- 1 root root  19M Aug 13  2025 KDDTrain.txt
-rw-r--r-- 1 root root 7.0M Aug 13  2025 nsl-kdd-dataset-filtered-version-of-kdd.zip
drwxr-xr-x 1 root root 4.0K May 21 13:27 sample_data


In [6]:
!find . -maxdepth 2 -type f

./.config/active_config
./.config/.last_opt_in_prompt.yaml
./.config/config_sentinel
./.config/default_configs.db
./.config/gce
./.config/.last_update_check.json
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/.last_survey_prompt.yaml
./KDDTrain_20Percent.arff
./KDDTest.txt
./KDDTrain_20Percent.txt
./KDDTest1.jpg
./KDDTest-21.txt
./nsl-kdd-dataset-filtered-version-of-kdd.zip
./index.html
./KDDTrain1.jpg
./KDDTrain.txt
./KDDTest-21.arff
./KDDTrain.arff
./KDDTest.arff
./sample_data/anscombe.json
./sample_data/README.md
./sample_data/mnist_test.csv
./sample_data/california_housing_train.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_test.csv


In [7]:
from pathlib import Path

train_path = Path("KDDTrain.txt")
test_path = Path("KDDTest.txt")

print("Train exists:", train_path.exists())
print("Test exists:", test_path.exists())

Train exists: True
Test exists: True


In [8]:
import pandas as pd

preview_train = pd.read_csv(train_path, header=None)
preview_test = pd.read_csv(test_path, header=None)

print("Train shape:", preview_train.shape)
print("Test shape:", preview_test.shape)

preview_train.head()

Train shape: (125973, 43)
Test shape: (22544, 43)


,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,40,41,42
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [10]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

In [11]:
train_path = Path("KDDTrain.txt")
test_path = Path("KDDTest.txt")

print("Train path:", train_path)
print("Test path:", test_path)
print("Train exists:", train_path.exists())
print("Test exists:", test_path.exists())

Train path: KDDTrain.txt
Test path: KDDTest.txt
Train exists: True
Test exists: True


In [12]:
KDD_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty"
]

TARGET_COLUMN = "label"
CATEGORICAL_COLUMNS = ["protocol_type", "service", "flag"]

FEATURE_COLUMNS = [
    col for col in KDD_COLUMNS
    if col not in ["label", "difficulty"]
]

LOG_COLUMNS = [
    "duration", "src_bytes", "dst_bytes", "hot", "num_compromised",
    "num_root", "num_file_creations", "count", "srv_count",
    "dst_host_count", "dst_host_srv_count"
]

NUMERIC_COLUMNS = [
    col for col in FEATURE_COLUMNS
    if col not in CATEGORICAL_COLUMNS
]

OTHER_NUMERIC_COLUMNS = [
    col for col in NUMERIC_COLUMNS
    if col not in LOG_COLUMNS
]

In [13]:
def load_kdd(path):
    df = pd.read_csv(path, header=None)

    if df.shape[1] == 43:
        df.columns = KDD_COLUMNS
    elif df.shape[1] == 42:
        df.columns = [col for col in KDD_COLUMNS if col != "difficulty"]
    else:
        raise ValueError(f"Expected 42 or 43 columns, got {df.shape[1]} from {path}")

    return df


def make_binary_target(df):
    df = df.copy()
    df["label"] = df["label"].astype(str).str.rstrip(".")
    df["label"] = (df["label"] != "normal").astype(int)
    return df

In [14]:
train_df = make_binary_target(load_kdd(train_path))
test_df = make_binary_target(load_kdd(test_path))

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]

X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain labels:")
print(y_train.value_counts())

print("\nTest labels:")
print(y_test.value_counts())

Train shape: (125973, 41)
Test shape: (22544, 41)

Train labels:
label
0    67343
1    58630
Name: count, dtype: int64

Test labels:
label
1    12833
0     9711
Name: count, dtype: int64


In [15]:
def build_preprocessor():
    log_numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
            ("scaler", StandardScaler())
        ]
    )

    regular_numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("log_numeric", log_numeric_pipeline, LOG_COLUMNS),
            ("regular_numeric", regular_numeric_pipeline, OTHER_NUMERIC_COLUMNS),
            ("categorical", categorical_pipeline, CATEGORICAL_COLUMNS)
        ]
    )

In [16]:
models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=250,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ),

    "xgboost": XGBClassifier(
        n_estimators=350,
        max_depth=7,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

In [17]:
def evaluate_model(name, pipeline, X_test, y_test):
    preds = pipeline.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()

    result = {
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "false_positive_rate": fp / (fp + tn) if (fp + tn) else 0,
        "true_positives": int(tp),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn)
    }

    if hasattr(pipeline, "predict_proba"):
        probs = pipeline.predict_proba(X_test)[:, 1]
        result["roc_auc"] = roc_auc_score(y_test, probs)

    return result

In [18]:
results = []
trained_pipelines = {}

for name, model in models.items():
    print(f"Training {name}...")

    pipeline = Pipeline(
        steps=[
            ("preprocess", build_preprocessor()),
            ("smote", SMOTE(random_state=42)),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    metrics = evaluate_model(name, pipeline, X_test, y_test)
    results.append(metrics)
    trained_pipelines[name] = pipeline

    print(pd.Series(metrics))
    print("-" * 60)

Training logistic_regression...
model                  logistic_regression
accuracy                          0.745298
precision                         0.913277
recall                            0.610535
f1                                0.731833
false_positive_rate               0.076614
true_positives                        7835
true_negatives                        8967
false_positives                        744
false_negatives                       4998
roc_auc                           0.791545
dtype: object
------------------------------------------------------------
Training random_forest...
model                  random_forest
accuracy                    0.782248
precision                   0.969098
recall                      0.637809
f1                          0.769303
false_positive_rate         0.026877
true_positives                  8185
true_negatives                  9450
false_positives                  261
false_negatives                 4648
roc_auc                 

In [19]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    by=["recall", "false_positive_rate", "f1"],
    ascending=[False, True, False]
)

,model,accuracy,precision,recall,f1,false_positive_rate,true_positives,true_negatives,false_positives,false_negatives,roc_auc
2,xgboost,0.791208,0.968303,0.654640,0.781161,0.028318,8401,9436,275,4432,0.965122
1,random_forest,0.782248,0.969098,0.637809,0.769303,0.026877,8185,9450,261,4648,0.961747
0,logistic_regression,0.745298,0.913277,0.610535,0.731833,0.076614,7835,8967,744,4998,0.791545


In [21]:
ranked = sorted(
    results,
    key=lambda row: (-row["recall"], row["false_positive_rate"], -row["f1"])
)

best_model_name = ranked[0]["model"]
best_pipeline = trained_pipelines[best_model_name]

print("Best model:", best_model_name)
print(json.dumps(ranked[0], indent=2))

Best model: xgboost
{
  "model": "xgboost",
  "accuracy": 0.791208303761533,
  "precision": 0.9683033656062702,
  "recall": 0.6546403802696174,
  "f1": 0.781161374308429,
  "false_positive_rate": 0.028318401812377717,
  "true_positives": 8401,
  "true_negatives": 9436,
  "false_positives": 275,
  "false_negatives": 4432,
  "roc_auc": 0.9651220394067103
}


In [22]:
probs = best_pipeline.predict_proba(X_test)[:, 1]

thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

threshold_results = []

for threshold in thresholds:
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "false_positive_rate": fp / (fp + tn),
        "false_negatives": fn,
        "false_positives": fp
    })

pd.DataFrame(threshold_results)

,threshold,precision,recall,f1,false_positive_rate,false_negatives,false_positives
0,0.5,0.968303,0.654640,0.781161,0.028318,4432,275
1,0.4,0.968307,0.659472,0.784592,0.028524,4370,277
2,0.3,0.968392,0.666095,0.789289,0.028730,4285,279
3,0.2,0.969020,0.682459,0.800878,0.028833,4075,280
4,0.1,0.967759,0.699369,0.811960,0.030790,3858,299


In [23]:
thresholds = np.arange(0.01, 0.21, 0.01)

threshold_results = []

probs = best_pipeline.predict_proba(X_test)[:, 1]

for threshold in thresholds:
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "false_positive_rate": fp / (fp + tn),
        "false_negatives": fn,
        "false_positives": fp
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.sort_values(
    by=["f1", "recall"],
    ascending=[False, False]
).head(10)

,threshold,precision,recall,f1,false_positive_rate,false_negatives,false_positives
0,0.01,0.966586,0.777683,0.861905,0.035527,2853,345
1,0.02,0.967063,0.755007,0.847978,0.033982,3144,330
2,0.03,0.967387,0.739656,0.838331,0.032952,3341,320
3,0.04,0.967363,0.725240,0.828984,0.032334,3526,314
4,0.05,0.967339,0.717759,0.824066,0.032026,3622,311
5,0.06,0.967291,0.712070,0.820287,0.031820,3695,309
6,0.07,0.967299,0.707629,0.817335,0.031614,3752,307
7,0.08,0.967759,0.704044,0.815102,0.030996,3798,301
8,0.09,0.967627,0.701083,0.813068,0.030996,3836,301
9,0.10,0.967759,0.699369,0.811960,0.030790,3858,299


In [28]:
from pathlib import Path
import json
import joblib
import numpy as np # Import numpy

Path("artifacts").mkdir(exist_ok=True)

CHOSEN_THRESHOLD = 0.1

deployment_artifact = {
    "pipeline": best_pipeline,
    "threshold": CHOSEN_THRESHOLD,
    "label_mapping": {
        0: "normal",
        1: "malicious"
    }
}

joblib.dump(best_pipeline, "artifacts/pipeline.pkl")
joblib.dump(deployment_artifact, "artifacts/ids_deployment_artifact.pkl")

def default_json_encoder(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

with open("artifacts/metrics.json", "w") as f:
    json.dump(ranked, f, indent=2, default=default_json_encoder)

with open("artifacts/threshold_metrics.json", "w") as f:
    json.dump(threshold_results, f, indent=2, default=default_json_encoder)

print("Saved artifacts/pipeline.pkl")
print("Saved artifacts/ids_deployment_artifact.pkl")
print("Saved artifacts/metrics.json")
print("Saved artifacts/threshold_metrics.json")

Saved artifacts/pipeline.pkl
Saved artifacts/ids_deployment_artifact.pkl
Saved artifacts/metrics.json
Saved artifacts/threshold_metrics.json


In [29]:
from google.colab import files

files.download("artifacts/pipeline.pkl")
files.download("artifacts/ids_deployment_artifact.pkl")
files.download("artifacts/metrics.json")
files.download("artifacts/threshold_metrics.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
loaded_pipeline = joblib.load("artifacts/pipeline.pkl")

sample = X_test.iloc[[0]]

prediction = loaded_pipeline.predict(sample)[0]
probability = loaded_pipeline.predict_proba(sample)[0][1]

print("Prediction:", "malicious" if prediction == 1 else "normal")
print("Malicious probability:", probability)

Prediction: malicious
Malicious probability: 0.99999785


In [3]:
import sklearn
import imblearn
import xgboost
import numpy
import pandas
import joblib

print("sklearn==", sklearn.__version__)
print("imbalanced-learn==", imblearn.__version__)
print("xgboost==", xgboost.__version__)
print("numpy==", numpy.__version__)
print("pandas==", pandas.__version__)
print("joblib==", joblib.__version__)

sklearn== 1.6.1
imbalanced-learn== 0.14.1
xgboost== 3.2.0
numpy== 2.0.2
pandas== 2.2.2
joblib== 1.5.3
